<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/%EB%B9%84%EC%A7%80%EB%8F%84%ED%95%99%EC%8A%B5_%EC%9E%90%EC%9C%A8%EC%A3%BC%ED%96%89%EC%9A%B4%EB%8F%99%ED%8C%A8%ED%84%B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install matplotlib seaborn scikit-learn


| 단계 | 설명                                          |
| -- | ------------------------------------------- |
| 1  | 운전 데이터를 **가짜로 생성** (속도, 가속도, 핸들 조작 각도)      |
| 2  | 데이터를 **표준화(정규화)**                           |
| 3  | **K-Means 알고리즘**으로 3가지 유형(고속/도심/정체)으로 자동 분류 |
| 4  | **시각화**로 각 클러스터(패턴)을 눈으로 확인                 |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 샘플 자율주행 데이터 생성
np.random.seed(42)
n_samples = 300

# # 고속도로 주행 데이터
# 속도 평균 90km/h, 가속도 낮음(0.5), 핸들 조작 작음(2도)
# 고속도로에서 비교적 일직선, 빠른 주행 특징
highway = np.random.normal(loc=[90, 0.5, 2], scale=[5, 0.2, 1], size=(100, 3))

#비슷하게 city(도심), traffic(정체) 데이터도 생성합니다:
city = np.random.normal(loc=[50, 1.5, 10], scale=[5, 0.3, 3], size=(100, 3))
traffic = np.random.normal(loc=[20, 0.2, 1], scale=[3, 0.1, 0.5], size=(100, 3))

data = np.vstack((highway, city, traffic))
df = pd.DataFrame(data, columns=['Speed(km/h)', 'Acceleration(m/s²)', 'Steering Angle'])

# 🔹 2단계: 데이터 정규화
# 정규화(표준화): 데이터 값들의 단위를 맞춰줌

# 이유: 어떤 값은 90(속도), 어떤 값은 0.5(가속도)처럼 범위가 달라서 공정한 비교가 안 됨
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

# 3단계: K-Means 클러스터링
# K=3: 우리는 3가지 패턴(고속, 도심, 정체)을 원한다고 설정

# K-Means가 데이터를 자동으로 3개의 그룹으로 나눔
#사람이 라벨을 붙이지 않아도, 머신이 비슷한 운전 스타일끼리 묶음
kmeans = KMeans(n_clusters=3, random_state=42)
df['Cluster'] = kmeans.fit_predict(scaled_data)

# 시각화
# 각 클러스터(분류된 그룹)를 색깔로 구분해서 보여줍니다

# 예: 파란 점은 고속도로, 빨간 점은 도심, 초록 점은 정체 운전
sns.pairplot(df, hue='Cluster', palette='Set1')
plt.suptitle('Driving Pattern Clustering Result', y=1.02)
plt.show()


🚘 비지도학습 예제 ②: 이상 운전 탐지 (Anomaly Detection)

| 단계           | 설명                           |
| ------------ | ---------------------------- |
| 🚗 운전 데이터 생성 | 정상 운전 + 비정상 운전 샘플 생성         |
| 🔍 모델 학습     | `Isolation Forest`로 이상치 탐지   |
| 📊 시각화       | 속도 vs 핸들각 그래프에서 빨간 점 = 이상 운전 |

🎯 결과 해석
Anomaly = -1이면 이상 운전 (빨간 점)

Anomaly = 1이면 정상 운전 (파란 점)

머신이 라벨 없이도 자동으로 위험 운전 탐지 🎯

좋아요! 이 그래프는 **비지도학습 기반 이상 운전 탐지 결과**를 시각화한 것입니다. 자세히 설명해드릴게요:

---

## 📊 그래프 해석: "운전 이상 탐지 결과"

### 🧾 X축: `Speed(km/h)`

* 차량의 주행 속도 (킬로미터/시간 단위)

### 🧾 Y축: `Steering Angle`

* 핸들 조향 각도 (도 단위, 클수록 급회전)

---

### 🔵 파란 점 (정상 운전, `Anomaly = 1`)

* 대부분의 점은 중앙에 밀집되어 있음
* 일반적인 속도 범위: 약 **40\~80km/h**
* 핸들 각도도 약 **1\~6도** 범위로, 일반적인 직진 또는 완만한 회전 운전

---

### 🔴 빨간 점 (이상 운전, `Anomaly = -1`)

총 5개:

| 위치 (속도, 핸들각) | 해석                                |
| ------------ | --------------------------------- |
| (100, 15)    | **고속 + 급회전**: 위험한 고속 급조향          |
| (20, 0.2)    | **너무 느림 + 거의 직진**: 정체 상태 또는 차량 이상 |
| (70, 30)     | **보통 속도 + 극단적 회전**: 급회전, 이상 조작    |
| (45, 20)     | **중속 + 급회전**: 급격한 핸들 조작 가능성       |
| (60, 12)     | **보통 속도 + 회전 심함**: 일반 범위를 벗어난 조향  |

---

## 🎯 핵심 요점

* 이 그래프는 **라벨 없이도 비정상 운전 패턴을 자동 탐지**한 결과예요.
* 사용한 알고리즘은 `Isolation Forest`, 이상치를 따로 구분해줍니다.
* 자율주행차나 운전 보조 시스템에서 **위험 감지, 사기 방지, 이상 운전 경고 시스템** 등에 활용 가능해요.

---

원하시면 이걸 **시간 순으로 표현하는 시계열 그래프**나, **LiDAR 데이터랑 융합한 응용 예제**도 만들어드릴 수 있어요.
궁금한 점 있으신가요? 😊


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest

# 1. 샘플 운전 데이터 생성
np.random.seed(42)

# 정상 운전: 속도 40~80km/h, 가속도 0.5~1.5, 핸들각 1~5도
normal_data = np.random.normal(loc=[60, 1.0, 3], scale=[10, 0.3, 1.0], size=(280, 3))

# 이상 운전: 급가속, 급회전 등 비정상값
anomaly_data = np.array([
    [100, 3.5, 15],   # 고속 급가속 급회전
    [20, 0.1, 0.2],   # 너무 느림
    [70, 5.0, 30],    # 급회전
    [45, 0.2, 20],    # 느린 속도 + 이상한 핸들각
    [60, 3.5, 12]     # 급가속
])

# 통합
data = np.vstack((normal_data, anomaly_data))
df = pd.DataFrame(data, columns=['Speed(km/h)', 'Acceleration', 'Steering Angle'])

# 2. Isolation Forest로 이상 탐지
model = IsolationForest(contamination=0.02, random_state=42)
df['Anomaly'] = model.fit_predict(df[['Speed(km/h)', 'Acceleration', 'Steering Angle']])

# 3. 시각화
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='Speed(km/h)', y='Steering Angle',
    hue='Anomaly', style='Anomaly',
    palette={1: 'blue', -1: 'red'}, s=100,
    data=df
)
plt.title("Anomaly Detection in Driving Behavior")
plt.legend(title='Anomaly (1=Normal, -1=Anomaly)')
plt.grid(True)
plt.show()
